# Olist SQL Data Preparation and Load Validation

## Objective

Prepare validated Olist datasets for loading into SQL Server while preserving table grain, identifiers, missing values, and monetary totals. Reconcile SQL results with the completed Python analyses.

## 1. Inspect Available Processed Datasets

Review the processed files before selecting source datasets and defining SQL tables. Detailed datasets will support SQL joins and aggregation, while existing analytical summaries will provide reference results for validation.

In [1]:
from pathlib import Path

# Locate the processed-data folder relative to the notebook's working directory.
processed_data_path = Path("../data/processed")

# Confirm the resolved location and folder availability.
print("Processed-data folder:", processed_data_path.resolve())
print("Folder exists:", processed_data_path.is_dir())

# List available files, including those inside analysis subfolders.
if processed_data_path.is_dir():
    processed_files = sorted(
        file_path
        for file_path in processed_data_path.rglob("*")
        if file_path.is_file()
    )

    print("\nTotal processed files:", len(processed_files))
    print("\nAvailable processed files:")

    for file_path in processed_files:
        print(file_path.relative_to(processed_data_path))
else:
    print("Check the notebook's working directory before continuing.")

Processed-data folder: C:\Users\Hp\DataAnalyst\data\processed
Folder exists: True

Total processed files: 23

Available processed files:
delivered_order_items_analysis_ready.parquet
olist_customers_clean.csv
olist_geolocation_clean.csv
olist_order_items_clean.csv
olist_order_payments_clean.csv
olist_order_reviews_clean.csv
olist_orders_clean.csv
olist_products_clean.csv
olist_sellers_clean.csv
payment_and_commercial_analysis\commercial_segment_summary.csv
payment_and_commercial_analysis\dominant_payment_method_summary.csv
payment_and_commercial_analysis\dominant_payment_value_summary.csv
payment_and_commercial_analysis\freight_burden_summary.csv
payment_and_commercial_analysis\installment_group_summary.csv
payment_and_commercial_analysis\multiple_payment_discrepancy_summary.csv
payment_and_commercial_analysis\order_commercial_payment_experience.parquet
payment_and_commercial_analysis\order_commercial_summary.parquet
payment_and_commercial_analysis\order_payment_reconciliation.parquet
p

## 2. Inspect Cleaned Source Tables for SQL

Inspect the dimensions and columns of the cleaned source tables before defining the SQL Server schema.

Preserve the source-table grain and retained quality flags. Apply the delivered-order scope explicitly in SQL queries and validate the results against the completed Python analyses.

Payment and review records may contain multiple rows per order. Their grain must be confirmed before defining keys and joining them to other tables.

In [2]:
import pandas as pd

# Map each SQL table name to its cleaned source file.
source_files = {
    'orders': 'olist_orders_clean.csv',
    'order_items': 'olist_order_items_clean.csv',
    'customers': 'olist_customers_clean.csv',
    'products': 'olist_products_clean.csv',
    'sellers': 'olist_sellers_clean.csv',
    'payments': 'olist_order_payments_clean.csv',
    'reviews': 'olist_order_reviews_clean.csv',
    'category_translation': 'product_category_name_translation_clean.csv'
}

# Load each source as strings for initial inspection, preserving leading zeros.
sql_source_tables = {}

for table_name, file_name in source_files.items():
    sql_source_tables[table_name] = pd.read_csv(processed_data_path / file_name, dtype='string')

# Display the name, shape, and columns of each source table.
for table_name, table in sql_source_tables.items():
    print(f"\nTable: {table_name}")
    print("Shape:", table.shape)
    print("Column names:", table.columns.tolist())


Table: orders
Shape: (99441, 17)
Column names: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'flag_delivered_missing_customer_date', 'flag_delivered_missing_carrier_date', 'flag_delivered_missing_approval_date', 'flag_delivered_missing_critical_timestamp', 'critical_timestamp_flag_count', 'flag_carrier_before_purchase', 'flag_customer_before_carrier', 'flag_carrier_before_approval', 'flag_lifecycle_timestamp_exception']

Table: order_items
Shape: (112650, 8)
Column names: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'flag_future_shipping_limit_date']

Table: customers
Shape: (99441, 5)
Column names: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Table: products
Shape: (32951, 15)
Column names: ['product_id', 'product_category_na

In [3]:
# Confirm that all eight source tables were loaded.
print("Loaded table count:", len(sql_source_tables))
print("Loaded table names:", list(sql_source_tables))

Loaded table count: 8
Loaded table names: ['orders', 'order_items', 'customers', 'products', 'sellers', 'payments', 'reviews', 'category_translation']


In [4]:
# Display the reviews table structure separately to avoid truncated output.
reviews_source = sql_source_tables['reviews']

print("\nReviews shape:", reviews_source.shape)
print("\nReviews columns:")

for column_name in reviews_source.columns:
    print(column_name)


Reviews shape: (99224, 9)

Reviews columns:
review_id
order_id
review_score
review_comment_title
review_comment_message
review_creation_date
review_answer_timestamp
review_response_duration
flag_review_response_over_30_days


## 3. Validate Candidate Keys Before Defining SQL Tables

Check candidate keys for missing values and duplicate combinations before assigning SQL Server primary keys.

Order items and payments use multi-column candidate keys because an order can contain several items and payment records. Review identifiers will be inspected individually and together with order identifiers before selecting a key.

These checks verify the exported source files and guide database constraints.

In [5]:
# Check whether the order and item identifiers uniquely identify each item row.
print("orders duplicated count: ", sql_source_tables['orders'].duplicated(subset=['order_id']).sum())
print("order_items duplicated count: ", sql_source_tables['order_items'].duplicated(subset=['order_id', 'order_item_id']).sum())
print("customers duplicated count: ", sql_source_tables['customers'].duplicated(subset=['customer_id']).sum())
print("products duplicated count: ", sql_source_tables['products'].duplicated(subset=['product_id']).sum())
print("sellers duplicated count: ", sql_source_tables['sellers'].duplicated(subset=['seller_id']).sum())
print("payments duplicated count: ", sql_source_tables['payments'].duplicated(subset=['order_id', 'payment_sequential']).sum())
print("category_translation duplicated count: ", sql_source_tables['category_translation'].duplicated(subset=['product_category_name']).sum())
print("reviews - review_id duplicated count: ", sql_source_tables['reviews'].duplicated(subset=['review_id']).sum())
print("reviews - review_id and order_id combination duplicated count: ", sql_source_tables['reviews'].duplicated(subset=['review_id', 'order_id']).sum())

print("\norders missing count: ", sql_source_tables['orders'][['order_id']].isna().sum())
print("order_items missing count: ", sql_source_tables['order_items'][['order_id', 'order_item_id']].isna().sum())
print("customers missing count: ", sql_source_tables['customers']['customer_id'].isna().sum())
print("products missing count: ", sql_source_tables['products']['product_id'].isna().sum())
print("sellers missing count: ", sql_source_tables['sellers']['seller_id'].isna().sum())
print("payments missing count: ", sql_source_tables['payments'][['order_id', 'payment_sequential']].isna().sum())
print("category_translation missing count: ", sql_source_tables['category_translation']['product_category_name'].isna().sum())
print("reviews - review_id missing count: ", sql_source_tables['reviews']['review_id'].isna().sum())
print("reviews - review_id and order_id combination missing count: ", sql_source_tables['reviews'][['review_id', 'order_id']].isna().sum())



orders duplicated count:  0
order_items duplicated count:  0
customers duplicated count:  0
products duplicated count:  0
sellers duplicated count:  0
payments duplicated count:  0
category_translation duplicated count:  0
reviews - review_id duplicated count:  814
reviews - review_id and order_id combination duplicated count:  0

orders missing count:  order_id    0
dtype: int64
order_items missing count:  order_id         0
order_item_id    0
dtype: int64
customers missing count:  0
products missing count:  0
sellers missing count:  0
payments missing count:  order_id              0
payment_sequential    0
dtype: int64
category_translation missing count:  0
reviews - review_id missing count:  0
reviews - review_id and order_id combination missing count:  review_id    0
order_id     0
dtype: int64


In [6]:

print("orders missing count: ", sql_source_tables['orders']['order_id'].isna().sum())

orders missing count:  0


### Candidate-Key Validation Summary

All selected candidate-key columns contain no missing values.

Orders, customers, products, sellers, and category translations have unique single-column keys. Order items are uniquely identified by the combination of order_id and order_item_id. Payment records are uniquely identified by order_id and payment_sequential.

The review_id column contains 814 duplicate occurrences beyond the first occurrence. However, the combination of review_id and order_id is unique and complete, making it suitable as the reviews table's composite primary key. All review records are retained.

In [7]:
# Count orders whose non-missing customer ID has no matching customer record.
customer_ids = sql_source_tables['orders']['customer_id']
print("Orders with unmatched customer IDs:", (customer_ids.notna() & ~customer_ids.isin(sql_source_tables['customers']['customer_id'])).sum())

order_ids = sql_source_tables['order_items']['order_id']
print("Orders_items with unmatched order IDs:", (order_ids.notna() & ~order_ids.isin(sql_source_tables['orders']['order_id'])).sum())

product_ids = sql_source_tables['order_items']['product_id']
print("Orders_items with unmatched product IDs:", (product_ids.notna() & ~product_ids.isin(sql_source_tables['products']['product_id'])).sum())

seller_ids = sql_source_tables['order_items']['seller_id']
print("Orders_items with unmatched seller IDs:", (seller_ids.notna() & ~seller_ids.isin(sql_source_tables['sellers']['seller_id'])).sum())


payments_order_ids = sql_source_tables['payments']['order_id']
print("payments with unmatched order IDs:", (payments_order_ids.notna() & ~payments_order_ids.isin(sql_source_tables['orders']['order_id'])).sum())

reviews_order_ids = sql_source_tables['reviews']['order_id']
print("reviews with unmatched order IDs:", (reviews_order_ids.notna() & ~reviews_order_ids.isin(sql_source_tables['orders']['order_id'])).sum())

product_category_names = sql_source_tables['products']['product_category_name']
print("products with unmatched  product_category_names:", (product_category_names.notna() & ~product_category_names.isin(sql_source_tables['category_translation']['product_category_name'])).sum())


Orders with unmatched customer IDs: 0
Orders_items with unmatched order IDs: 0
Orders_items with unmatched product IDs: 0
Orders_items with unmatched seller IDs: 0
payments with unmatched order IDs: 0
reviews with unmatched order IDs: 0
products with unmatched  product_category_names: 13


In [8]:
# Count missing child identifiers separately from unmatched identifiers.
relationship_columns = {
    'orders.customer_id': customer_ids,
    'order_items.order_id': order_ids,
    'order_items.product_id': product_ids,
    'order_items.seller_id': seller_ids,
    'payments.order_id': payments_order_ids,
    'reviews.order_id': reviews_order_ids,
    'products.product_category_name': product_category_names
}

for relationship_name, values in relationship_columns.items():
    print(f"{relationship_name} — missing count: {values.isna().sum()}")

# Identify products with a non-missing category that has no translation.
unmatched_category_mask = product_category_names.notna() & ~product_category_names.isin(sql_source_tables['category_translation']['product_category_name'])

# Display distinct untranslated category names and their product counts.
print("\nUnmatched categories and product counts:")
display(product_category_names.loc[unmatched_category_mask].value_counts().rename_axis('product_category_name').reset_index(name='product_count'))

orders.customer_id — missing count: 0
order_items.order_id — missing count: 0
order_items.product_id — missing count: 0
order_items.seller_id — missing count: 0
payments.order_id — missing count: 0
reviews.order_id — missing count: 0
products.product_category_name — missing count: 610

Unmatched categories and product counts:


,product_category_name,product_count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


### Relationship Validation Summary

The six core relationships between orders, customers, order items, products, sellers, payments, and reviews contain no missing or unmatched child identifiers. These relationships are suitable for foreign-key constraints.

Product categories contain 610 missing values and 13 product records without matching translations. The untranslated categories are portateis_cozinha_e_preparadores_de_alimentos (10 products) and pc_gamer (3 products).

All products are retained. Category translations will be joined using a left join, without enforcing a foreign-key constraint on product_category_name. Existing category-analysis fields and quality flags are preserved.

## 4. Inspect Source Values for SQL Data Types

Inspect identifier and text lengths, missing-value counts, and quality-flag values before defining SQL Server column types.

Identifiers and ZIP-code prefixes will remain text to preserve their formatting. Monetary values will use fixed-point decimal types. Numeric, timestamp, and boolean conversions will be validated before loading.

In [9]:
# Build a column profile for all eight source tables.
column_profile_rows = []

for table_name, table in sql_source_tables.items():
    for column_name in table.columns:
        column_profile_rows.append({
            'table_name': table_name,
            'column_name': column_name,
            'missing_value_count': table[column_name].isna().sum(),
            'maximum_string_length': table[column_name].str.len().max()
        })

sql_column_profile = pd.DataFrame(column_profile_rows)

# Display the complete profile without truncating rows or column names.
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(sql_column_profile)


,table_name,column_name,missing_value_count,maximum_string_length
0,orders,order_id,0,32
1,orders,customer_id,0,32
2,orders,order_status,0,11
3,orders,order_purchase_timestamp,0,19
4,orders,order_approved_at,160,19
5,orders,order_delivered_carrier_date,1783,19
6,orders,order_delivered_customer_date,2965,19
7,orders,order_estimated_delivery_date,0,10
8,orders,flag_delivered_missing_customer_date,0,5
9,orders,flag_delivered_missing_carrier_date,0,5


In [10]:
# Inspect the non-missing values used to represent each quality flag.
flag_profile_rows = []

for table_name, table in sql_source_tables.items():
    for column_name in table.columns:
        if column_name.startswith('flag_'):
            flag_profile_rows.append({
                'table_name': table_name,
                'flag_column': column_name,
                'missing_value_count': table[column_name].isna().sum(),
                'distinct_values': sorted(
                    table[column_name].dropna().unique().tolist()
                )
            })

sql_flag_profile = pd.DataFrame(flag_profile_rows)

# Display every flag and its observed string values before conversion.
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(sql_flag_profile)

,table_name,flag_column,missing_value_count,distinct_values
0,orders,flag_delivered_missing_customer_date,0,"[False, True]"
1,orders,flag_delivered_missing_carrier_date,0,"[False, True]"
2,orders,flag_delivered_missing_approval_date,0,"[False, True]"
3,orders,flag_delivered_missing_critical_timestamp,0,"[False, True]"
4,orders,flag_carrier_before_purchase,0,"[False, True]"
5,orders,flag_customer_before_carrier,0,"[False, True]"
6,orders,flag_carrier_before_approval,0,"[False, True]"
7,orders,flag_lifecycle_timestamp_exception,0,"[False, True]"
8,order_items,flag_future_shipping_limit_date,0,"[False, True]"
9,products,flag_missing_catalog_attributes,0,"[False, True]"


In [11]:
# Confirm that the column profile includes every source column exactly once.
expected_column_count = sum(table.shape[1] for table in sql_source_tables.values())

print("Tables profiled:", sql_column_profile['table_name'].nunique())
print("Columns profiled:", len(sql_column_profile))
print("Column count reconciles:", len(sql_column_profile) == expected_column_count)
print("Duplicate table-column combinations:", sql_column_profile.duplicated(subset=['table_name', 'column_name']).sum())

Tables profiled: 8
Columns profiled: 74
Column count reconciles: True
Duplicate table-column combinations: 0


### Source-Column and Quality-Flag Inspection Summary

The eight source tables contain 74 columns, including 22 quality flags. All quality flags contain only the string values 'False' and 'True', with no missing values. Quality flags are retained as the text values 'False' and 'True' in SQL Server, using NVARCHAR(5) columns.

Identifiers and ZIP-code prefixes will remain text. Missing timestamps, product attributes, review comments, and reference states will be preserved as SQL NULL values.

The 623 missing English product-category names comprise 610 products with missing source categories and 13 products without matching translations. Numeric ranges and timestamp formats will be checked before finalizing SQL data types.

## 5. Validate Numeric Fields Before SQL Conversion

Check numeric fields for parsing failures, minimum and maximum values, and fractional values before selecting SQL data types.

Missing source values will remain missing. Monetary values will use fixed-point decimal types, while integer types will be selected only for fields confirmed to contain whole numbers. Source tables remain unchanged during inspection.

In [12]:
# Specify the numeric columns to inspect in each source table.
numeric_columns = {
    'orders': ['critical_timestamp_flag_count'],
    'order_items': ['order_item_id', 'price', 'freight_value'],
    'products': [
        'product_name_length',
        'product_description_length',
        'product_photos_qty',
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ],
    'payments': [
        'payment_sequential',
        'payment_installments',
        'payment_value'
    ],
    'reviews': ['review_score']
}

numeric_profile_rows = []

# Convert temporary series for inspection without changing the source tables.
for table_name, column_names in numeric_columns.items():
    for column_name in column_names:
        source_column = sql_source_tables[table_name][column_name]
        numeric_values = pd.to_numeric(source_column, errors='coerce')

        # Distinguish existing missing values from numeric parsing failures.
        parsing_failure_mask = source_column.notna() & numeric_values.isna()

        # Count available numeric values with a non-zero fractional part.
        available_values = numeric_values.dropna()
        fractional_value_count = available_values.mod(1).ne(0).sum()

        numeric_profile_rows.append({
            'table_name': table_name,
            'column_name': column_name,
            'missing_source_values': source_column.isna().sum(),
            'parsing_failures': parsing_failure_mask.sum(),
            'minimum_value': numeric_values.min(),
            'maximum_value': numeric_values.max(),
            'fractional_value_count': fractional_value_count
        })

sql_numeric_profile = pd.DataFrame(numeric_profile_rows)

# Display the complete numeric profile without rounding its values.
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None
):
    display(sql_numeric_profile)

# Validate profile coverage and check for numeric parsing failures.
print("Numeric columns profiled:", len(sql_numeric_profile))
print(
    "Duplicate table-column combinations:",
    sql_numeric_profile.duplicated(
        subset=['table_name', 'column_name']
    ).sum()
)
print(
    "Total parsing failures:",
    sql_numeric_profile['parsing_failures'].sum()
)
print(
    "All numeric columns parse successfully:",
    sql_numeric_profile['parsing_failures'].eq(0).all()
)

,table_name,column_name,missing_source_values,parsing_failures,minimum_value,maximum_value,fractional_value_count
0,orders,critical_timestamp_flag_count,0,0,0.00,2.00,0
1,order_items,order_item_id,0,0,1.00,21.00,0
2,order_items,price,0,0,0.85,6735.00,79862
3,order_items,freight_value,0,0,0.00,409.68,111015
4,products,product_name_length,610,0,5.00,76.00,0
5,products,product_description_length,610,0,4.00,3992.00,0
6,products,product_photos_qty,610,0,1.00,20.00,0
7,products,product_weight_g,2,0,0.00,40425.00,0
8,products,product_length_cm,2,0,7.00,105.00,0
9,products,product_height_cm,2,0,2.00,105.00,0


Numeric columns profiled: 15
Duplicate table-column combinations: 0
Total parsing failures: 0
All numeric columns parse successfully: True


In [13]:
# Inspect the original review-duration values before choosing a conversion.
review_duration_source = sql_source_tables['reviews']['review_response_duration']

print("Source dtype:", review_duration_source.dtype)
print("Missing source values:", review_duration_source.isna().sum())

# Display ten original values without modifying the source table.
print("\nSample review-response durations:")
display(review_duration_source.head(10))

Source dtype: string
Missing source values: 0

Sample review-response durations:


0    0 days 21:46:59
1    1 days 03:05:13
2    1 days 14:36:24
3    0 days 22:02:06
4    1 days 10:26:53
5    3 days 00:39:37
6    2 days 19:30:34
7    0 days 21:36:06
8    1 days 12:05:37
9    1 days 16:45:47
Name: review_response_duration, dtype: string

### 5.1 Review-Response Duration Validation

Review-response durations are stored as strings containing days, hours, minutes, and seconds. They require timedelta parsing rather than direct numeric conversion.

Inspect parsing completeness, duration ranges, and fractional seconds before preparing a numeric duration field for SQL. Preserve the original source values.

In [14]:
# Parse duration strings into a temporary timedelta series.
review_duration_parsed = pd.to_timedelta(
    review_duration_source,
    errors='coerce'
)

# Identify non-missing source values that could not be parsed.
duration_parsing_failures = (
    review_duration_source.notna()
    & review_duration_parsed.isna()
)

# Express durations in total seconds, including the days component.
review_duration_seconds = review_duration_parsed.dt.total_seconds()

# Validate parsing and inspect the range before selecting a SQL type.
print("Duration records:", len(review_duration_parsed))
print("Parsing failures:", duration_parsing_failures.sum())
print("Missing parsed durations:", review_duration_parsed.isna().sum())
print("Minimum duration:", review_duration_parsed.min())
print("Maximum duration:", review_duration_parsed.max())
print("Negative durations:", review_duration_seconds.lt(0).sum())
print(
    "Durations with fractional seconds:",
    review_duration_seconds.dropna().mod(1).ne(0).sum()
)

# Compare the original strings with their total-second equivalents.
display(pd.DataFrame({
    'original_duration': review_duration_source,
    'duration_seconds': review_duration_seconds
}).head())


Duration records: 99224
Parsing failures: 0
Missing parsed durations: 0
Minimum duration: 0 days 02:08:29
Maximum duration: 518 days 16:46:52
Negative durations: 0
Durations with fractional seconds: 0


,original_duration,duration_seconds
0,0 days 21:46:59,78419.0
1,1 days 03:05:13,97513.0
2,1 days 14:36:24,138984.0
3,0 days 22:02:06,79326.0
4,1 days 10:26:53,124013.0


### Review-Response Duration Validation Summary

All 99,224 review-response durations parsed successfully, with no missing values, negative durations, or fractional seconds.

Durations range from 2 hours, 8 minutes, and 29 seconds to 518 days, 16 hours, 46 minutes, and 52 seconds. Whole-second storage preserves the observed precision.

Review-response durations were converted to total seconds in a temporary series for validation only. The SQL Server table retains the original review_response_duration strings in an NVARCHAR(50) column. Long response durations and their existing quality flags are preserved.

## 6. Validate Date and Timestamp Fields

Check that non-missing date and timestamp strings can be converted without introducing new missing values. Inspect their ranges before defining SQL date and datetime columns.

Existing missing timestamps and previously flagged exceptions will be preserved.

In [15]:
# Define the date and timestamp columns to inspect in each table.
timestamp_columns = {
    'orders': [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ],
    'order_items': ['shipping_limit_date'],
    'reviews': ['review_creation_date', 'review_answer_timestamp']
}

timestamp_profile_rows = []

# Parse temporary series without modifying the source tables.
for table_name, column_names in timestamp_columns.items():
    for column_name in column_names:
        source_column = sql_source_tables[table_name][column_name]
        parsed_values = pd.to_datetime(source_column, errors='coerce')

        # Identify non-missing source values that failed conversion.
        parsing_failure_mask = source_column.notna() & parsed_values.isna()

        # Record missing counts, parsing failures, and timestamp ranges.
        timestamp_profile_rows.append({
            'table_name': table_name,
            'column_name': column_name,
            'missing_source_values': source_column.isna().sum(),
            'parsing_failures': parsing_failure_mask.sum(),
            'missing_after_parsing': parsed_values.isna().sum(),
            'earliest_timestamp': parsed_values.min(),
            'latest_timestamp': parsed_values.max()
        })

sql_timestamp_profile = pd.DataFrame(timestamp_profile_rows)

# Display the complete timestamp profile.
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None
):
    display(sql_timestamp_profile)


,table_name,column_name,missing_source_values,parsing_failures,missing_after_parsing,earliest_timestamp,latest_timestamp
0,orders,order_purchase_timestamp,0,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,orders,order_approved_at,160,0,160,2016-09-15 12:16:38,2018-09-03 17:40:06
2,orders,order_delivered_carrier_date,1783,0,1783,2016-10-08 10:34:01,2018-09-11 19:48:28
3,orders,order_delivered_customer_date,2965,0,2965,2016-10-11 13:46:32,2018-10-17 13:22:46
4,orders,order_estimated_delivery_date,0,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00
5,order_items,shipping_limit_date,0,0,0,2016-09-19 00:15:34,2020-04-09 22:35:08
6,reviews,review_creation_date,0,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00
7,reviews,review_answer_timestamp,0,0,0,2016-10-07 18:32:28,2018-10-29 12:27:35


In [16]:
# Validate profile coverage and conversion completeness.
print("Timestamp columns profiled:", len(sql_timestamp_profile))
print(
    "Duplicate table-column combinations:",
    sql_timestamp_profile.duplicated(
        subset=['table_name', 'column_name']
    ).sum()
)
print(
    "Total parsing failures:",
    sql_timestamp_profile['parsing_failures'].sum()
)
print(
    "Missing counts match for every column:",
    sql_timestamp_profile['missing_source_values'].eq(
        sql_timestamp_profile['missing_after_parsing']
    ).all()
)

Timestamp columns profiled: 8
Duplicate table-column combinations: 0
Total parsing failures: 0
Missing counts match for every column: True


### Date and Timestamp Validation Summary

All eight date and timestamp columns parsed successfully, with no new missing values introduced.

Existing missing timestamps were preserved: 160 approval timestamps, 1,783 carrier-delivery timestamps, and 2,965 customer-delivery timestamps.

The shipping-limit range extends to April 2020. Previously flagged date exceptions remain in the source data, together with their quality flags.

## 7. Validate Identifier Formats and Monetary Precision

Confirm identifier lengths and ZIP-code-prefix formats before choosing text column types. Verify that monetary values can be stored at two-decimal precision without changing their values.

These checks guide the SQL schema while preserving the source data.

In [17]:
from decimal import Decimal, InvalidOperation

# Check the length of every identifier column except the item sequence number.
identifier_profile_rows = []

for table_name, table in sql_source_tables.items():
    for column_name in table.columns:
        if column_name.endswith('_id') and column_name != 'order_item_id':
            source_values = table[column_name]
            invalid_length_mask = (
                source_values.notna()
                & source_values.str.len().ne(32)
            ).fillna(False)

            identifier_profile_rows.append({
                'table_name': table_name,
                'column_name': column_name,
                'missing_count': source_values.isna().sum(),
                'invalid_length_count': invalid_length_mask.sum(),
                'all_non_missing_ids_have_32_characters':
                    not invalid_length_mask.any()
            })

sql_identifier_profile = pd.DataFrame(identifier_profile_rows)
display(sql_identifier_profile)


# Check that ZIP-code prefixes contain exactly five ASCII digits.
zip_columns = {
    'customers': 'customer_zip_code_prefix',
    'sellers': 'seller_zip_code_prefix'
}

zip_profile_rows = []

for table_name, column_name in zip_columns.items():
    source_values = sql_source_tables[table_name][column_name]
    valid_format_mask = source_values.str.fullmatch(r'[0-9]{5}', na=False)
    invalid_format_mask = source_values.notna() & ~valid_format_mask

    zip_profile_rows.append({
        'table_name': table_name,
        'column_name': column_name,
        'missing_count': source_values.isna().sum(),
        'invalid_format_count': invalid_format_mask.sum(),
        'every_value_has_five_digits': valid_format_mask.all()
    })

    # Display missing or invalid prefixes without modifying them.
    exception_mask = ~valid_format_mask
    if exception_mask.any():
        print(f"\nZIP-prefix exceptions in {table_name}:")
        display(
            sql_source_tables[table_name].loc[
                exception_mask,
                ['customer_id' if table_name == 'customers' else 'seller_id',
                 column_name]
            ]
        )

sql_zip_profile = pd.DataFrame(zip_profile_rows)
display(sql_zip_profile)


# Classify monetary strings using exact decimal arithmetic.
def check_cent_precision(value):
    if pd.isna(value):
        return 'missing'

    try:
        decimal_value = Decimal(value)

        if not decimal_value.is_finite():
            return 'invalid'

        # Equality allows trailing zeros but rejects any required rounding.
        if decimal_value == decimal_value.quantize(Decimal('0.01')):
            return 'exact_to_cents'

        return 'requires_rounding'

    except InvalidOperation:
        return 'invalid'


# Verify two-decimal precision for item prices, freight, and payments.
monetary_columns = {
    'order_items': ['price', 'freight_value'],
    'payments': ['payment_value']
}

monetary_profile_rows = []

for table_name, column_names in monetary_columns.items():
    for column_name in column_names:
        source_values = sql_source_tables[table_name][column_name]
        precision_status = source_values.map(check_cent_precision)

        monetary_profile_rows.append({
            'table_name': table_name,
            'column_name': column_name,
            'missing_count': precision_status.eq('missing').sum(),
            'invalid_value_count': precision_status.eq('invalid').sum(),
            'requires_rounding_count':
                precision_status.eq('requires_rounding').sum(),
            'every_value_exact_to_cents':
                precision_status.eq('exact_to_cents').all()
        })

        # Display invalid values or values requiring rounding for investigation.
        exception_mask = precision_status.isin(
            ['invalid', 'requires_rounding']
        )
        if exception_mask.any():
            print(f"\nMonetary exceptions in {table_name}.{column_name}:")
            display(source_values.loc[exception_mask].to_frame())

sql_monetary_precision_profile = pd.DataFrame(monetary_profile_rows)
display(sql_monetary_precision_profile)

,table_name,column_name,missing_count,invalid_length_count,all_non_missing_ids_have_32_characters
0,orders,order_id,0,0,True
1,orders,customer_id,0,0,True
2,order_items,order_id,0,0,True
3,order_items,product_id,0,0,True
4,order_items,seller_id,0,0,True
5,customers,customer_id,0,0,True
6,customers,customer_unique_id,0,0,True
7,products,product_id,0,0,True
8,sellers,seller_id,0,0,True
9,payments,order_id,0,0,True


,table_name,column_name,missing_count,invalid_format_count,every_value_has_five_digits
0,customers,customer_zip_code_prefix,0,0,True
1,sellers,seller_zip_code_prefix,0,0,True


,table_name,column_name,missing_count,invalid_value_count,requires_rounding_count,every_value_exact_to_cents
0,order_items,price,0,0,0,True
1,order_items,freight_value,0,0,0,True
2,payments,payment_value,0,0,0,True


In [18]:
# Validate identifier-profile coverage, completeness, and lengths.
print("Identifier profile rows:", len(sql_identifier_profile))
print("Identifier profile has 12 rows:", len(sql_identifier_profile) == 12)
print("Missing identifiers:", sql_identifier_profile['missing_count'].sum())
print(
    "Invalid identifier lengths:",
    sql_identifier_profile['invalid_length_count'].sum()
)
print(
    "All non-missing identifiers have 32 characters:",
    sql_identifier_profile['all_non_missing_ids_have_32_characters'].all()
)

# Validate ZIP-prefix coverage and report missing or invalid values.
print("\nZIP-prefix profile rows:", len(sql_zip_profile))
print("ZIP-prefix profile has 2 rows:", len(sql_zip_profile) == 2)
print("Missing ZIP prefixes:", sql_zip_profile['missing_count'].sum())
print(
    "Invalid ZIP-prefix formats:",
    sql_zip_profile['invalid_format_count'].sum()
)
print(
    "Every ZIP prefix has five digits:",
    sql_zip_profile['every_value_has_five_digits'].all()
)

# Validate monetary-profile coverage and exact two-decimal representation.
print("\nMonetary profile rows:", len(sql_monetary_precision_profile))
print(
    "Monetary profile has 3 rows:",
    len(sql_monetary_precision_profile) == 3
)
print(
    "Missing monetary values:",
    sql_monetary_precision_profile['missing_count'].sum()
)
print(
    "Invalid monetary values:",
    sql_monetary_precision_profile['invalid_value_count'].sum()
)
print(
    "Monetary values requiring rounding:",
    sql_monetary_precision_profile['requires_rounding_count'].sum()
)
print(
    "Every monetary value is exact to cents:",
    sql_monetary_precision_profile['every_value_exact_to_cents'].all()
)

Identifier profile rows: 12
Identifier profile has 12 rows: True
Missing identifiers: 0
Invalid identifier lengths: 0
All non-missing identifiers have 32 characters: True

ZIP-prefix profile rows: 2
ZIP-prefix profile has 2 rows: True
Missing ZIP prefixes: 0
Invalid ZIP-prefix formats: 0
Every ZIP prefix has five digits: True

Monetary profile rows: 3
Monetary profile has 3 rows: True
Missing monetary values: 0
Invalid monetary values: 0
Monetary values requiring rounding: 0
Every monetary value is exact to cents: True


### Identifier and Monetary Precision Validation Summary

All 12 identifier columns checked contain complete, 32-character values. The order_item_id sequence is excluded from this text-identifier check.

Customer and seller ZIP-code prefixes contain exactly five digits, with no missing or invalid values. They will remain text to preserve leading zeros.

Item prices, freight values, and payment values contain no missing or invalid entries and can be represented exactly at two-decimal precision without rounding. These monetary fields will use DECIMAL columns in SQL Server.

### Preparation Checkpoint

Source profiling is complete for eight cleaned tables, covering candidate
keys, relationships, missing values, numeric and timestamp parsing,
identifier formats, and monetary precision.

Quality flags and review-response duration are retained as text in the
current SQL Server import.

Full-table monetary reconciliation between Python and SQL remains pending.
SQL import checks are maintained in the separate SQL validation script.